# Triage Results
Script/NB for generating the lookup of the triage results. For every project in
the test set of each of ASF, EF, OF, and GH we will predict on the best model of
each. Later, we can run the triage model and pick the output label.

Output columns will be:
- Incubator
- Project
- Outcome
- ASF Pred
- EF Pred
- OF Pred
- GF Pred
- Triage Incubator Pred (to be done later)

***

## Env Setup

In [1]:
import decalfc as pex
from decalfc.abstractions.modeldata import *
from decalfc.abstractions.tsmodel import *
from decalfc.pipeline.inference import *

import numpy as np

/home/aashok17/miniconda3/envs/pex/lib/python3.11/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


INFO: Pandarallel will run on 6 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [2]:
incs = ["apache", "eclipse", "osgeo", "github"]
model_archs = dict(zip(incs, ["BLSTM", "BLSTM", "DLSTM", "DLSTM"]))

***
## Finding the Median Fold Projects

In [3]:
def find_median_fold(inc: str, k: int=5, model_arch: str="BLSTM", **kwargs) -> tuple[set[str], set[str]]:
    """
    Finds the median fold from the cross-validation, i.e. the projects
    themselves that we used to generate the CV results reported in the paper.
    """
    
    # === train the models for this fold === #
    # load all data in a folds iterator
    folds = ModelData.gen_k_folds(
        transfer_strategy=f"{inc} --> {inc}",
        transform_kwargs=kwargs.get("transform-kwargs", dict()),
        nfolds=k,
        yield_projs=True
    )
    
    # store only the median trial in the perf database; we'll create a temporary
    # perf-db for this kfold trial, pick the median, and update the actual perf
    # db
    temp_perf_path = f""
    pfd = PerfData(temp_perf_path)
    fold_projs: list[tuple[set[str], set[str]]] = list()
    perfs = list()

    # train model & test for each fold
    for fold, projs in folds:
        ## grab sample tensor, i.e. first tensor we have
        sample_tensor = fold.tensors["train"]["x"][0]
            
        ## ensure some hyperparams
        hyperparams = {"input_size": sample_tensor.shape[1]}
        hyperparams.update(kwargs.get("hyperparams", dict()))

        ## build model
        model = TimeSeriesModel(
            model_arch=model_arch,
            hyperparams=hyperparams
        )
        
        ## train & test
        print(len(fold.tensors["train"]["y"]), len(fold.tensors["test"]["y"]))
        print(len(projs[0]), len(projs[1]))
        model.train(fold)
        model.test(fold)
        
        ## track perf
        pfd._add_entry(
            transfer_strat=f"{inc} --> {inc}",
            model_arch=model_arch,
            preds=model.preds,
            targets=model.targets,
            export_db=False
        )
        perfs.append(pfd.data[(pfd.data.metric == "f1-score") & (pfd.data.label == "weighted avg")]["perf"].iloc[-1])
        
        ## add projs
        fold_projs.append(projs)
    
    # median finding
    median_idx = np.where(perfs == np.median(perfs))[0][0]

    # export train and test for median fold
    return fold_projs[median_idx]

In [4]:
median_folds = {
    inc: find_median_fold(inc=inc[0].upper(), model_arch=model_archs[inc])
    for inc in incs
}

/mnt/data1/Arjun/pex-forecaster/decalfc/abstractions/netdata.py:388: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_data = self.data[self.data["proj_name"].isin(project_subset)].groupby("proj_name").apply(process_group)



<Decoded Transfer Strategy>
Log [0.0h, 0.0m, 0.000s]> Original: `a --> a`
Options selected for train set:
Log [0.0h, 0.0m, 0.000s]> 	apache dataset, version tech: 1, social: 1 with no augmentations using 0.00% of the data reserved for training
Options selected for test set:
Log [0.0h, 0.0m, 0.000s]> 	apache dataset, version tech: 1, social: 1 with no augmentations using 100.00% of the data reserved for testing
Log [0.0h, 0.0m, 0.000s]> Setting up netdata
Log [0.0h, 0.0m, 0.000s]> Reading in/generating data
Log [0.0h, 0.0m, 0.016s]> Generating project status, split
Log [0.0h, 0.0m, 0.000s]> Generating data lookup
Log [0.0h, 0.0m, 0.069s]> Generating tensors

<Tensor Info For Apache>


100%|██████████| 263/263 [00:00<00:00, 57636.34it/s]


<Decoded Transfer Strategy>
Log [0.0h, 0.0m, 0.000s]> Original: `a --> a`
Options selected for train set:
Log [0.0h, 0.0m, 0.000s]> 	apache dataset, version tech: 1, social: 1 with no augmentations using 0.00% of the data reserved for training
Options selected for test set:
Log [0.0h, 0.0m, 0.000s]> 	apache dataset, version tech: 1, social: 1 with no augmentations using 100.00% of the data reserved for testing

<Model Setup>
using ***cpu*** for training...

<Model Chosen :: Bidirectional Lstm>


211 52
211 52


...

Log [0.0h, 0.0m, 1.212s]> Epoch [19/100] | loss: 0.0679, test loss: 0.0866, test acc: 0.8000 | lr: 0.000500


100%|██████████| 16/16 [00:01<00:00, 15.86it/s]


Log [0.0h, 0.0m, 1.077s]> Epoch [20/100] | loss: 0.0679, test loss: 0.0866, test acc: 0.8000 | lr: 0.000500


100%|██████████| 16/16 [00:00<00:00, 16.14it/s]

Log [0.0h, 0.0m, 1.068s]> Epoch [21/100] | loss: 0.0679, test loss: 0.0866, test acc: 0.8000 | lr: 0.000500
Log [0.0h, 0.0m, 0.000s]> Early stopping triggered. loading best model weights.
Log [0.0h, 0.0m, 0.001s]> Training completed.
Model Name: DLSTM
Input size: 14
Hidden size: 64
Number of layers: 2
Shape of X_train: torch.Size([103, 14])


In [5]:
median_folds = {
    inc: (set(f[0]), set(f[1]))
    for inc, f in median_folds.items()
}

In [6]:
from json import dumps
print(list(map(len, median_folds["apache"])))
print(list(map(len, median_folds["eclipse"])))
print(list(map(len, median_folds["osgeo"])))
print(list(map(len, median_folds["github"])))

[211, 52]
[108, 31]
[16, 4]
[16, 5]


In [7]:
print(median_folds)

{'apache': ({'skywalking', 'dubbo', 'opennlp', 'photark', 'ibatis', 'mesos', 'batchee', 'hise', 'daffodil', 'manifoldcf', 'carbondata', 'curator', 'shindig', 'samoa', 'mod_ftp', 'droids', 'rocketmq', 'zeta', 'fineract', 'trafficserver', 'cloudstack', 'airflow', 'parquet', 'lucy', 'xap', 'mrql', 'trafodion', 'directmemory', 'pulsar', 'ripple', 'jdo', 'tamaya', 'edgent', 'nuvem', 'guacamole', 'stonehenge', 'devicemap', 'roller', 'directory', 'nmaven', 'weex', 'plc4x', 'superset', 'iota', 'derby', 'bigtop', 'rya', 'storm', 'ctakes', 'ratis', 'openjpa', 'jackrabbit', 'druid', 'crunch', 'concerted', 'myriad', 'commonsrdf', 'lens', 'struts', 'thrift', 'click', 'etch', 'empire', 'imperius', 'sentry', 'flex', 'airavata', 'stratos', 'apex', 'npanday', 'lokahi', 'marmotta', 'solr', 'servicecomb', 'vxquery', 'metamodel', 'bval', 'esme', 'oozie', 'any23', 'wicket', 'subversion', 'aries', 'openwebbeans', 'clerezza', 'juneau', 'pig', 'tephra', 'singa', 'pubscribe', 'sqoop', 'pivot', 'sling', 'lucene

***
## Inference on the Median Folds

In [8]:
def train_helper(m: TimeSeriesModel, X, y):
    # track losses
    losses = {}
    test_losses = {}
    best_loss = float("inf")
    best_epoch = 0
    patience = 10
    TOLERANCE = 1e-4
    
    # initialize optimizer and scheduler
    m.optimizer = torch.optim.AdamW(
        m.model.parameters(),
        lr=m.hyperparams["learning_rate"],
        weight_decay=0.01
    )
    m.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        m.optimizer, mode="min", factor=0.5, patience=10
    )

    m.loss_fc = FocalLoss(alpha=0.5, gamma=2.0)

    # Training loop
    for epoch in range(m.hyperparams["num_epochs"]):
        m.model.train()
        losses[epoch] = []

        # stochastic GD
        for data, target in tqdm(list(zip(X, y))):
            # shape into (B --> 1, ntimesteps, nfeatures)
            data = data.to(m.device).reshape(1, data.shape[0], -1)
            target = target.to(m.device).to(torch.float32)

            # forward pass
            pred = m.model.predict(data)[..., 1].to(torch.float32)

            # compute loss (Focal Loss)
            loss = m.loss_fc(pred, target)
            
            # backward pass
            m.optimizer.zero_grad()
            loss.backward()

            # grad clipping
            torch.nn.utils.clip_grad_norm_(m.model.parameters(), max_norm=1.0)

            # step the optimizer
            m.optimizer.step()

            # loss tracking
            losses[epoch].append(loss.item())

        # mean loss
        losses[epoch] = np.mean(losses[epoch])

        # logging
        current_lr = m.optimizer.param_groups[0]["lr"]
        log(f"Epoch [{epoch + 1}/{m.hyperparams['num_epochs']}] | "
            f"Loss: {losses[epoch]:.4f}, "
            f"LR: {current_lr:.6f}", "log")

        # lr scheduling
        if m.scheduler is not None:
            m.scheduler.step(losses[epoch])

        # early stopping protocol
        avg_loss = losses[epoch]

        if avg_loss < best_loss - TOLERANCE:
            best_loss = avg_loss
            best_model_weights = copy.deepcopy(m.model.state_dict())      
            patience = 10
            best_epoch = epoch
        else:
            patience -= 1
            if patience == 0:
                log("Early stopping triggered. Loading best model weights.", "log")
                m.model.load_state_dict(best_model_weights)
                break

    # check for divergence
    if np.isnan(sorted(losses.items(), reverse=True)[0][1]) or np.isinf(sorted(losses.items(), reverse=True)[0][1]):
        log("NaN or Inf loss generated, i.e. failed to converge: ignoring and exiting", "error")

    log("Training completed.", "log")

    print(f"Model Name: {m.model_arch}")
    print(f"Input size: {m.hyperparams['input_size']}")
    print(f"Hidden size: {m.hyperparams['hidden_size']}")
    print(f"Number of layers: {m.hyperparams['num_layers']}")
    print(f"Shape of X_train: {X[2].shape}")

In [9]:
def netdata_inference(nd: NetData, model: TimeSeriesModel, test_set: set[str]):
    """
    Inference on the test set of the NetData
    """
    
    # tracker
    predictions = dict()
    predictions["project"] = list()
    predictions["prediction"] = list()
    predictions["target"] = list()
    
    # eval
    model.model.eval()
    with torch.no_grad():
        # iterate projects
        for proj_name, X in nd.data_dict.items():
            # skip
            if proj_name not in test_set:
                continue
            
            # convert to tensor
            X = torch.tensor(X).to(model.device)
            X = X.reshape(1, X.shape[0], -1)
            y_true = (
                "graduated" if proj_name in nd.project_status["graduated"] else
                ("retired" if proj_name in nd.project_status["retired"] else "incubating")
            )
            
            # predictions
            out = model.model(X)
            pred_label = torch.argmax(out, dim=1)
            y_pred = pred_label.cpu().numpy()[0]
            y_pred = "graduated" if y_pred == 1 else "retired"
            
            # update trackers
            predictions["project"].append(proj_name)
            predictions["prediction"].append(y_pred)
            predictions["target"].append(y_true)
    
    # export
    return predictions

def fold_inference(inc: str, folds: dict[str, tuple[set[str], set[str]]], model_arch: str="BLSTM", **kwargs):
    """
    Trains a model on the given train projects and outputs labels for all test
    projects.
    """
    
    # build the train and test sets for all incubators
    nds = {
        i: NetData(
            incubator=i,
            split_set={"train": folds[i][0], "test": folds[i][1]},
            is_train="both",
            verbose=False,
        )
        for i in incs
    }
    
    # setup model & train on the train set
    model = TimeSeriesModel(
        model_arch=model_arch,
        **kwargs
    )
    train_helper(
        model,
        X=nds[inc].tensors["train"]["x"],
        y=nds[inc].tensors["train"]["y"],
    )
    
    # inference on each project in the nd
    return {
        i: netdata_inference(nds[i], model, test_set=folds[i][1])
        for i in incs
    }

In [10]:
# grab pred results
res = {
    inc: fold_inference(inc, median_folds, model_archs[inc])
    for inc in incs
}

Log [0.0h, 0.0m, 0.359s]> Setting up netdata
Log [0.0h, 0.0m, 0.000s]> Reading in/generating data


/mnt/data1/Arjun/pex-forecaster/decalfc/abstractions/netdata.py:388: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_data = self.data[self.data["proj_name"].isin(project_subset)].groupby("proj_name").apply(process_group)


Log [0.0h, 0.0m, 0.027s]> Generating project status, split
Log [0.0h, 0.0m, 0.000s]> Generating data lookup
Log [0.0h, 0.0m, 0.106s]> Generating tensors

<Tensor Info For Apache>


100%|██████████| 263/263 [00:00<00:00, 75560.10it/s]

Log [0.0h, 0.0m, 0.032s]> Setting up netdata
Log [0.0h, 0.0m, 0.000s]> Reading in/generating data
Log [0.0h, 0.0m, 0.011s]> Generating project status, split
Log [0.0h, 0.0m, 0.000s]> Generating data lookup


...

Log [0.0h, 0.0m, 0.902s]> Epoch [18/100] | loss: 0.0726, lr: 0.001000


100%|██████████| 16/16 [00:00<00:00, 17.41it/s]


Log [0.0h, 0.0m, 0.921s]> Epoch [19/100] | loss: 0.0726, lr: 0.001000


100%|██████████| 16/16 [00:00<00:00, 19.54it/s]


Log [0.0h, 0.0m, 0.820s]> Epoch [20/100] | loss: 0.0726, lr: 0.001000
Log [0.0h, 0.0m, 0.000s]> Early stopping triggered. loading best model weights.
Log [0.0h, 0.0m, 0.001s]> Training completed.
Model Name: DLSTM
Input size: 14
Hidden size: 64
Number of layers: 2
Shape of X_train: torch.Size([171, 14])


In [11]:
import pandas as pd

res = {
    inc: {
        i: pd.DataFrame(res[inc][i])
        for i in res[inc]
    }
    for inc in incs
}
res

{'apache': {'apache':              project prediction     target
  0           accumulo  graduated  graduated
  1              alois    retired    retired
  2          amaterasu    retired    retired
  3          asterixdb  graduated  graduated
  4                awf    retired    retired
  5            beehive  graduated  graduated
  6               blur    retired    retired
  7           brooklyn  graduated  graduated
  8            calcite  graduated  graduated
  9            cayenne  graduated  graduated
  10        deltacloud  graduated  graduated
  11  dolphinscheduler  graduated  graduated
  12             eagle  graduated  graduated
  13           echarts  graduated  graduated
  14            falcon  graduated  graduated
  15             flume  graduated  graduated
  16        freemarker  graduated  graduated
  17         ftpserver    retired  graduated
  18             geode  graduated  graduated
  19           gobblin  graduated  graduated
  20            gossip    retired  

***
## Organizing Results

In [12]:
def transform_inc_res(inc_res, inc: str):
    for i, df in inc_res.items():
        df["incubator"] = i
        df.rename(columns={"prediction": f"{inc}_pred"}, inplace=True)
        
    return pd.concat(inc_res.values(), ignore_index=True)

In [13]:
r1 = [transform_inc_res(r, i) for i, r in res.items()]
r2 = pd.concat(r1, axis=1, join="outer")
r3 = r2.loc[:, ~r2.columns.duplicated()]
r3

,project,apache_pred,target,incubator,eclipse_pred,osgeo_pred,github_pred
0,accumulo,graduated,graduated,apache,graduated,graduated,retired
1,alois,retired,retired,apache,graduated,retired,retired
2,amaterasu,retired,retired,apache,graduated,retired,retired
3,asterixdb,graduated,graduated,apache,graduated,graduated,retired
4,awf,retired,retired,apache,graduated,graduated,retired
...,...,...,...,...,...,...,...
87,busybox,retired,graduated,github,graduated,retired,retired
88,django-social-auth,retired,retired,github,graduated,retired,retired
89,inherited_resources,retired,retired,github,graduated,retired,retired
90,scripted,retired,retired,github,graduated,retired,retired


In [14]:
df = r3[["incubator", "project", "target", "apache_pred", "eclipse_pred", "osgeo_pred", "github_pred"]]
df.to_csv("triage_results.csv", index=False)

In [15]:
df

,incubator,project,target,apache_pred,eclipse_pred,osgeo_pred,github_pred
0,apache,accumulo,graduated,graduated,graduated,graduated,retired
1,apache,alois,retired,retired,graduated,retired,retired
2,apache,amaterasu,retired,retired,graduated,retired,retired
3,apache,asterixdb,graduated,graduated,graduated,graduated,retired
4,apache,awf,retired,retired,graduated,graduated,retired
...,...,...,...,...,...,...,...
87,github,busybox,graduated,retired,graduated,retired,retired
88,github,django-social-auth,retired,retired,graduated,retired,retired
89,github,inherited_resources,retired,retired,graduated,retired,retired
90,github,scripted,retired,retired,graduated,retired,retired


In [18]:
f = {i: (list(f[0]), list(f[1])) for i, f in median_folds.items()}
f

{'apache': (['skywalking',
   'dubbo',
   'opennlp',
   'photark',
   'ibatis',
   'mesos',
   'batchee',
   'hise',
   'daffodil',
   'manifoldcf',
   'carbondata',
   'curator',
   'shindig',
   'samoa',
   'mod_ftp',
   'droids',
   'rocketmq',
   'zeta',
   'fineract',
   'trafficserver',
   'cloudstack',
   'airflow',
   'parquet',
   'lucy',
   'xap',
   'mrql',
   'trafodion',
   'directmemory',
   'pulsar',
   'ripple',
   'jdo',
   'tamaya',
   'edgent',
   'nuvem',
   'guacamole',
   'stonehenge',
   'devicemap',
   'roller',
   'directory',
   'nmaven',
   'weex',
   'plc4x',
   'superset',
   'iota',
   'derby',
   'bigtop',
   'rya',
   'storm',
   'ctakes',
   'ratis',
   'openjpa',
   'jackrabbit',
   'druid',
   'crunch',
   'concerted',
   'myriad',
   'commonsrdf',
   'lens',
   'struts',
   'thrift',
   'click',
   'etch',
   'empire',
   'imperius',
   'sentry',
   'flex',
   'airavata',
   'stratos',
   'apex',
   'npanday',
   'lokahi',
   'marmotta',
   'solr',
 

In [ ]:
import json

with open("triage_median_folds.json", "w") as file:
    json.dump(f, file, indent=4)